# 3.1 Coverage of protected areas and other effective area-based conservation measures

In [1]:
!pip install pyproj>=3.5.0
!pip install geopandas
!pip install fiona
!pip install gdown
!pip install tabulate
!pip install -U "folium>=0.12" branca matplotlib mapclassify


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.7/341.7 kB 5.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 7.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 30.9 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.6/444.6 kB 9.0 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for fiona: filename=fiona-1.10.1-cp311-cp311-linux_aarch64.whl size=1185024 sha256=cb3cc7cbce61f08070eca2431ebbd7e7b46172cade58da17ac14b7472d83fa58
  Stored in directory: /home/jovyan/.cache/pip/wheels/db/52/85/fd6a71a1d256467a4957402ab923fa39e45a280878a11cbf35
Successfully built fiona
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 11.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62

# Capa Espacial Áreas Protegidas SIMBIO

In [2]:
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, MultiPoint, LineString, MultiLineString, Polygon, MultiPolygon, LinearRing
import gdown 
from pathlib import Path
import zipfile
from IPython.display import Markdown, display
from pyproj import Geod
import shapely

geod = Geod(ellps="WGS84")
    
LAYER_AP_URLS = [
    "https://arcgis.mma.gob.cl/server/rest/services/SIMBIO/SIMBIO_AP/FeatureServer/0", #Áreas Protegidas
    "https://arcgis.mma.gob.cl/server/rest/services/SIMBIO/SIMBIO_AP/FeatureServer/1", #Otras Designaciones
    "https://arcgis.mma.gob.cl/server/rest/services/SIMBIO/SIMBIO_AP/FeatureServer/2", #Sitios Prioritarios
    "https://arcgis.mma.gob.cl/server/rest/services/SIMBIO/SIMBIO_AP/FeatureServer/3", #Conservación Privada
]

SESSION = requests.Session()

def geodesic_area_m2(geom):
    if geom is None or geom.is_empty:
        return 0.0
    # pyproj devuelve área firmada (signo depende de orientación); usamos valor absoluto
    area, _perim = geod.geometry_area_perimeter(geom)
    return abs(area)

def _get_json(url, params, timeout=180):
    r = SESSION.get(url, params=params, timeout=timeout)
    r.raise_for_status()
    js = r.json()
    if "error" in js:
        raise RuntimeError(js["error"])
    return js

def _get_layer_info(layer_url: str) -> dict:
    return _get_json(layer_url, {"f": "pjson"}, timeout=60)

def _get_object_ids(layer_url: str, where="1=1") -> list[int]:
    query_url = layer_url.rstrip("/") + "/query"
    js = _get_json(query_url, {
        "f": "pjson",
        "where": where,
        "returnIdsOnly": "true",
        "returnGeometry": "false",
    }, timeout=120)
    return js.get("objectIds") or []

def _ring_area(coords):
    a = 0.0
    for (x1, y1), (x2, y2) in zip(coords, coords[1:]):
        a += (x1 * y2 - x2 * y1)
    return a / 2.0

def _esri_polygon_to_shape(rings):
    if not rings:
        return None

    norm = []
    for ring in rings:
        if not ring:
            continue
        if ring[0] != ring[-1]:
            ring = ring + [ring[0]]
        norm.append(ring)

    outers, holes = [], []
    for ring in norm:
        try:
            lr = LinearRing(ring)
            if lr.is_ccw:
                holes.append(ring)
            else:
                outers.append(ring)
        except Exception:
            if _ring_area(ring) < 0:
                outers.append(ring)
            else:
                holes.append(ring)

    if not outers and norm:
        areas = [(abs(_ring_area(r)), r) for r in norm]
        areas.sort(reverse=True, key=lambda t: t[0])
        outers = [areas[0][1]]
        holes = [r for _, r in areas[1:]]

    outer_polys = [Polygon(o) for o in outers if len(o) >= 4]
    if not outer_polys:
        return None

    holes_by_outer = {i: [] for i in range(len(outer_polys))}
    for h in holes:
        hp = Polygon(h)
        if hp.is_empty:
            continue
        pt = hp.representative_point()
        for i, op in enumerate(outer_polys):
            if op.contains(pt):
                holes_by_outer[i].append(h)
                break

    polys = []
    for i, o in enumerate(outers):
        try:
            polys.append(Polygon(o, holes=holes_by_outer[i]))
        except Exception:
            polys.append(Polygon(o))

    return polys[0] if len(polys) == 1 else MultiPolygon(polys)

def _esri_geom_to_shape(geom: dict, geom_type: str):
    if geom is None:
        return None

    gt = (geom_type or "").lower()

    if "point" in gt:
        x, y = geom.get("x"), geom.get("y")
        return Point(x, y) if x is not None and y is not None else None

    if "multipoint" in gt:
        pts = geom.get("points") or []
        return MultiPoint([Point(x, y) for x, y in pts]) if pts else None

    if "polyline" in gt:
        paths = geom.get("paths") or []
        lines = [LineString(p) for p in paths if p and len(p) >= 2]
        if not lines:
            return None
        return lines[0] if len(lines) == 1 else MultiLineString(lines)

    if "polygon" in gt:
        rings = geom.get("rings") or []
        return _esri_polygon_to_shape(rings)

    return None

def _query_by_object_ids(layer_url, object_ids, out_sr=4326, out_fields="*", max_allowable_offset=None, timeout=180):
    query_url = layer_url.rstrip("/") + "/query"
    params = {
        "f": "pjson",
        "objectIds": ",".join(map(str, object_ids)),
        "outFields": out_fields,
        "returnGeometry": "true",
        "outSR": out_sr,
        "returnZ": "false",
        "returnM": "false",
    }
    if max_allowable_offset is not None:
        # Generaliza geometría en unidades del SR de salida (en 4326 es grados)
        params["maxAllowableOffset"] = max_allowable_offset

    js = _get_json(query_url, params, timeout=timeout)
    return js.get("features") or []

def _fetch_features_resilient(layer_url, ids, geom_type, out_sr=4326, timeout=180):
    """
    Intenta bajar features por IDs.
    Si falla (500 u otros), parte en 2 recursivamente.
    En caso extremo de un solo ID fallando, reintenta con generalización; si sigue fallando, lo salta.
    """
    if not ids:
        return []

    try:
        return _query_by_object_ids(layer_url, ids, out_sr=out_sr, max_allowable_offset=None, timeout=timeout)
    except Exception:
        # Si el lote es grande, bisección
        if len(ids) > 1:
            mid = len(ids) // 2
            left = _fetch_features_resilient(layer_url, ids[:mid], geom_type, out_sr=out_sr, timeout=timeout)
            right = _fetch_features_resilient(layer_url, ids[mid:], geom_type, out_sr=out_sr, timeout=timeout)
            return left + right

        # Caso: un solo OBJECTID está rompiendo el query
        bad_id = ids[0]
        # Intento 1: generalización suave (en grados, si out_sr=4326)
        for offset in (1e-5, 5e-5, 1e-4, 5e-4, 1e-3):
            try:
                feats = _query_by_object_ids(layer_url, [bad_id], out_sr=out_sr, max_allowable_offset=offset, timeout=timeout)
                return feats
            except Exception:
                continue

        print(f"[WARN] No se pudo descargar geometría para OBJECTID={bad_id} en {layer_url}. Se omite.")
        return []

def fetch_arcgis_layer_gdf(layer_url: str, where="1=1", chunk_size=100, out_sr=4326, timeout=180) -> gpd.GeoDataFrame:
    info = _get_layer_info(layer_url)
    geom_type = info.get("geometryType", "")
    oids = _get_object_ids(layer_url, where=where)

    if not oids:
        return gpd.GeoDataFrame(geometry=[], crs=f"EPSG:{out_sr}")

    rows = []
    for i in range(0, len(oids), chunk_size):
        ids_chunk = oids[i:i + chunk_size]
        feats = _fetch_features_resilient(layer_url, ids_chunk, geom_type, out_sr=out_sr, timeout=timeout)

        for ft in feats:
            attrs = ft.get("attributes") or {}
            geom = ft.get("geometry")
            attrs["geometry"] = _esri_geom_to_shape(geom, geom_type)
            rows.append(attrs)

    gdf = gpd.GeoDataFrame(rows, geometry="geometry", crs=f"EPSG:{out_sr}")
    gdf = gdf[~gdf.geometry.isna()].copy()
    return gdf

# -------------------------
# Descargar + unir capas
# -------------------------
gdfs = [fetch_arcgis_layer_gdf(u, chunk_size=100) for u in LAYER_AP_URLS]
areas_protegidas_capa_gpd = gpd.GeoDataFrame(gdfs[0], geometry="geometry", crs=gdfs[0].crs)
areas_protegidas_capa_gpd = areas_protegidas_capa_gpd[areas_protegidas_capa_gpd["designacion"]!="Reserva de la Biófera"]
otras_designaciones_capa_gpd = gpd.GeoDataFrame(gdfs[1], geometry="geometry", crs=gdfs[1].crs)
sitios_prioritarios_capa_gpd = gpd.GeoDataFrame(gdfs[1], geometry="geometry", crs=gdfs[2].crs)
conservacion_privada_capa_gpd = gpd.GeoDataFrame(gdfs[2], geometry="geometry", crs=gdfs[3].crs)


In [34]:
areas_protegidas_capa_gpd["area_m2"] = areas_protegidas_capa_gpd.geometry.apply(geodesic_area_m2)
areas_protegidas_capa_gpd["area_ha"] = areas_protegidas_capa_gpd["area_m2"] / 10_000
areas_protegidas_capa_gpd_area_total_ha = areas_protegidas_capa_gpd["area_ha"].sum()
areas_protegidas_capa_gpd_area_total_km2 = areas_protegidas_capa_gpd_area_total_ha / 100

otras_designaciones_capa_gpd["area_m2"] = otras_designaciones_capa_gpd.geometry.apply(geodesic_area_m2)
otras_designaciones_capa_gpd["area_ha"] = otras_designaciones_capa_gpd["area_m2"] / 10_000
otras_designaciones_capa_gpd_area_total_ha = otras_designaciones_capa_gpd["area_ha"].sum()
otras_designaciones_capa_gpd_area_total_km2 = otras_designaciones_capa_gpd_area_total_ha / 100

sitios_prioritarios_capa_gpd["area_m2"] = sitios_prioritarios_capa_gpd.geometry.apply(geodesic_area_m2)
sitios_prioritarios_capa_gpd["area_ha"] = sitios_prioritarios_capa_gpd["area_m2"] / 10_000
sitios_prioritarios_capa_gpd_area_total_ha = sitios_prioritarios_capa_gpd["area_ha"].sum()
sitios_prioritarios_capa_gpd_area_total_km2 = sitios_prioritarios_capa_gpd_area_total_ha / 100

conservacion_privada_capa_gpd["area_m2"] = conservacion_privada_capa_gpd.geometry.apply(geodesic_area_m2)
conservacion_privada_capa_gpd["area_ha"] = conservacion_privada_capa_gpd["area_m2"] / 10_000
conservacion_privada_capa_gpd_area_total_ha = conservacion_privada_capa_gpd["area_ha"].sum()
conservacion_privada_capa_gpd_area_total_km2 = conservacion_privada_capa_gpd_area_total_ha / 100

areas_protegidas_total_gpd = gpd.GeoDataFrame(
    pd.concat([areas_protegidas_capa_gpd,otras_designaciones_capa_gpd,conservacion_privada_capa_gpd], ignore_index=True), 
    geometry="geometry", crs=gdfs[0].crs)

print("CRS (entrada):", areas_protegidas_total_gpd.crs)
print("Filas:", len(areas_protegidas_total_gpd))

areas_protegidas_total_gpd["geometry"] = areas_protegidas_total_gpd.geometry.make_valid()
areas_protegidas_total_gpd_union = areas_protegidas_total_gpd.geometry.union_all()

areas_protegidas_total_gpd["area_m2"] = areas_protegidas_total_gpd.geometry.apply(geodesic_area_m2)
areas_protegidas_total_gpd["area_ha"] = areas_protegidas_total_gpd["area_m2"] / 10_000
areas_protegidas_total_gpd_area_total_ha = areas_protegidas_total_gpd["area_ha"].sum()
areas_protegidas_total_gpd_area_total_km2 = areas_protegidas_total_gpd_area_total_ha / 100

CRS (entrada): EPSG:4326
Filas: 675


# Capa Espacial Localización de Humedales

In [23]:
LAYER_HUMEDAL_URL = "https://arcgis.mma.gob.cl/server/rest/services/SIMBIO/SIMBIO_HUMEDALES/FeatureServer/1"

humedales_gpd = fetch_arcgis_layer_gdf(LAYER_HUMEDAL_URL, chunk_size=100)

humedales_gpd["area_m2"] = humedales_gpd.geometry.apply(geodesic_area_m2)
humedales_gpd["area_ha"] = humedales_gpd["area_m2"] / 10_000
humedales_gpd_area_total_ha = humedales_gpd["area_ha"].sum()
humedales_gpd_area_total_km2 = humedales_gpd_area_total_ha / 100



# Regiones de Chile

In [4]:
from pathlib import Path
import zipfile
chile_url='https://www.bcn.cl/obtienearchivo?id=repositorio/10221/10398/2/Regiones.zip'
zip_path = Path("Regiones.zip")
referer = "https://www.bcn.cl/siit/mapas_vectoriales"
headers = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36",
    "Referer": referer,
    "Accept": "*/*",
    "Accept-Language": "es-CL,es;q=0.9,en;q=0.8",
    "Connection": "keep-alive",
}
s = requests.Session()
# 1) “Visita” la página para obtener cookies si las piden
s.get(referer, headers=headers, timeout=60)
# 2) Descarga el ZIP
with s.get(chile_url, headers=headers, stream=True, timeout=120, allow_redirects=True) as r:
    r.raise_for_status()
    with open(zip_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

print("Descargado:", zip_path)
with zipfile.ZipFile(zip_path, "r") as z:
    names = z.namelist()
    shp_files = [n for n in names if n.lower().endswith(".shp")]
    print("SHP encontrados:", shp_files)
shp_inside = shp_files[0]  # elige el correcto si hay más de uno
regiones_gpd = gpd.read_file(f"zip://{zip_path}!{shp_inside}")
regiones_gpd = regiones_gpd.to_crs(epsg=4326)
print("CRS:", regiones_gpd.crs)
regiones_gpd["area_m2"] = regiones_gpd.geometry.apply(geodesic_area_m2)
regiones_gpd["area_ha"] = regiones_gpd["area_m2"] / 10_000
regiones_gpd_area_total_ha = regiones_gpd["area_ha"].sum()
regiones_gpd_area_total_km2 = regiones_gpd_area_total_ha / 100


Descargado: Regiones.zip
SHP encontrados: ['Regional.shp']
CRS: EPSG:4326


# Ecosistemas Marinos

In [6]:

ecosistemas_marinos_url='https://lineasdebasepublicas.mma.gob.cl/datos_abiertos/dataset/c144e690-f266-49a9-bb74-a85214a95164/resource/99283f8c-3d6b-4b14-aa82-4d859ad1fece/download/ecosistemas-marinos_geojson.zip'

download_path = Path("marinos_geojson.zip")
extract_dir = Path("marinos_geojson")

response = requests.get(ecosistemas_marinos_url)
response.raise_for_status()

download_path.write_bytes(response.content)
print(f"ZIP descargado en: {download_path.resolve()}")

extract_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(download_path, "r") as zf:
    zf.extractall(extract_dir)
    print("Contenido del ZIP:")
    for name in zf.namelist():
        print(" -", name)

geojson_files = list(extract_dir.rglob("*.geojson"))
if not geojson_files:
    raise FileNotFoundError("No se encontró ningún archivo .geojson en el ZIP")

geojson_path = geojson_files[0]
print(f"GeoJSON encontrado: {geojson_path}")

ecosistemas_marinos_gpd = gpd.read_file(geojson_path)
print("CRS:", ecosistemas_marinos_gpd.crs)
ecosistemas_marinos_union = ecosistemas_marinos_gpd.geometry.union_all()

ecosistemas_marinos_gpd_area_total_m2 = geodesic_area_m2(ecosistemas_marinos_union)
ecosistemas_marinos_gpd_area_total_ha = ecosistemas_marinos_gpd_area_total_m2 / 10_000
ecosistemas_marinos_gpd_area_total_km2 = ecosistemas_marinos_gpd_area_total_ha / 100

ZIP descargado en: /home/jovyan/7NR-Indicators/indicadores/3_1_percentage_protected_area_coverage/marinos_geojson.zip
Contenido del ZIP:
 - Ecosistemas Marinos.geojson
GeoJSON encontrado: marinos_geojson/Ecosistemas Marinos.geojson
CRS: EPSG:4326


# Chile

In [7]:
def fix_geoms(gdf):
    gdf = gdf.copy()
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty]

    # reparar
    if hasattr(gdf.geometry, "make_valid"):
        gdf["geometry"] = gdf.geometry.make_valid()
    else:
        gdf["geometry"] = gdf.geometry.buffer(0)

    # filtra lo que siga malo
    gdf = gdf[gdf.is_valid & gdf.geometry.notna() & ~gdf.geometry.is_empty]
    return gdf

regiones_fix = fix_geoms(regiones_gpd)
marinos_fix  = fix_geoms(ecosistemas_marinos_gpd)


In [8]:
chile_gpd = gpd.GeoDataFrame(
    pd.concat([regiones_fix, marinos_fix], ignore_index=True),
    crs=regiones_fix.crs
)

chile_union = chile_gpd.geometry.union_all()

chile_gpd_area_total_m2 = geodesic_area_m2(chile_union)
chile_gpd_area_total_ha = chile_gpd_area_total_m2 / 10_000
chile_gpd_area_total_km2 = chile_gpd_area_total_ha / 100

# Cálculo Indicador

## Cobertura Total

In [10]:
porcentaje = (areas_protegidas_total_gpd_area_total_km2 / chile_gpd_area_total_km2) * 100
print(f"Superficie Chile: {chile_gpd_area_total_km2:,.0f} km²")
print(f"Superficie Áreas Protegidas: {areas_protegidas_total_gpd_area_total_km2:,.0f} km²")
print(f"% protegido: {porcentaje:.2f}%")

Superficie Chile: 4,324,229 km²
Superficie Áreas Protegidas: 1,995,850 km²
% protegido: 46.16%


## Cobertura por Ecosistemas Marinos

In [17]:
areas_protegidas_marinas_gpd = gpd.overlay(ecosistemas_marinos_gpd, areas_protegidas_total_gpd, how="intersection", keep_geom_type=False)
areas_protegidas_marinas_gpd["geometry"] = areas_protegidas_marinas_gpd.geometry.make_valid()
areas_protegidas_marinas_gpd_union = areas_protegidas_marinas_gpd.geometry.union_all()
areas_protegidas_marinas_gpd_area_total_m2 = geodesic_area_m2(areas_protegidas_marinas_gpd_union)
areas_protegidas_marinas_gpd_area_total_ha = areas_protegidas_marinas_gpd_area_total_m2 / 10_000
areas_protegidas_marinas_gpd_area_total_km2 = areas_protegidas_marinas_gpd_area_total_ha / 100
porcentaje = (areas_protegidas_marinas_gpd_area_total_km2 / ecosistemas_marinos_gpd_area_total_km2) * 100
print(f"Superficie Ecosistemas Marinos: {ecosistemas_marinos_gpd_area_total_km2:,.0f} km²")
print(f"Superficie Áreas Protegidas Marinas: {areas_protegidas_marinas_gpd_area_total_km2:,.0f} km²")
print(f"% protegido: {porcentaje:.2f}%")

Superficie Ecosistemas Marinos: 3,571,362 km²
Superficie Áreas Protegidas Marinas: 1,501,391 km²
% protegido: 42.04%


## Cobertura por Ecosistemas Terrestres

In [19]:
areas_protegidas_terrestres_gpd = gpd.overlay(regiones_gpd, areas_protegidas_total_gpd, how="intersection", keep_geom_type=False)
areas_protegidas_terrestres_gpd["geometry"] = areas_protegidas_terrestres_gpd.geometry.make_valid()
areas_protegidas_terrestres_gpd_union = areas_protegidas_terrestres_gpd.geometry.union_all()
areas_protegidas_terrestres_gpd_area_total_m2 = geodesic_area_m2(areas_protegidas_terrestres_gpd_union)
areas_protegidas_terrestres_gpd_area_total_ha = areas_protegidas_terrestres_gpd_area_total_m2 / 10_000
areas_protegidas_terrestres_gpd_area_total_km2 = areas_protegidas_terrestres_gpd_area_total_ha / 100
porcentaje = (areas_protegidas_terrestres_gpd_area_total_km2 / regiones_gpd_area_total_km2) * 100
print(f"Superficie Regiones Chile: {regiones_gpd_area_total_km2:,.0f} km²")
print(f"Superficie Áreas Protegidas Terrestres: {areas_protegidas_terrestres_gpd_area_total_km2:,.0f} km²")
print(f"% protegido: {porcentaje:.2f}%")

Superficie Regiones Chile: 758,686 km²
Superficie Áreas Protegidas Terrestres: 287,062 km²
% protegido: 37.84%


## Desagregaciones Requeridas
El indicador final se debe presentar desglosado para cubrir los requisitos de la Meta 3:
## Cobertura por Tipo (AP vs. OECM):
Calcular el porcentaje de superficie total cubierto por Áreas Protegidas (AP).
### Cobertura AP
Áreas protegidas (AP): Parque Nacional, Parque Marino, Reserva Nacional, Reserva Marina, Reserva Forestal, Monumento Natural, Santuarios de la Naturaleza, Área de Conservación de Múltiples Usos

In [20]:
porcentaje = (areas_protegidas_total_gpd_area_total_km2 / chile_gpd_area_total_km2) * 100
print(f"Superficie Chile: {chile_gpd_area_total_km2:,.0f} km²")
print(f"Superficie Áreas Protegidas: {areas_protegidas_total_gpd_area_total_km2:,.0f} km²")
print(f"% protegido: {porcentaje:.2f}%")

Superficie Chile: 4,324,229 km²
Superficie Áreas Protegidas: 1,995,850 km²
% protegido: 46.16%


### Cobertura OECM
Calcular el porcentaje de superficie total cubierto por Otras Medidas de Conservación Eficaces Basadas en Áreas (OECM).
Otras medidas eficaces de conservación (OECM):, Paisaje de conservación, Conservación Privada y Comunitaria, Bien nacional protegido, humedales urbanos

In [25]:
# Unión OECM 

oecm_total_gpd = gpd.GeoDataFrame(
    pd.concat([otras_designaciones_capa_gpd, conservacion_privada_capa_gpd, humedales_gpd], ignore_index=True),
    crs=otras_designaciones_capa_gpd.crs
)
oecm_total_gpd["geometry"] = oecm_total_gpd.geometry.make_valid()
oecm_total_gpd_union = oecm_total_gpd.geometry.union_all()
oecm_total_gpd_area_total_m2 = geodesic_area_m2(oecm_total_gpd_union)
oecm_total_gpd_area_total_ha = oecm_total_gpd_area_total_m2 / 10_000
oecm_total_gpd_area_total_km2 = oecm_total_gpd_area_total_ha / 100
porcentaje = (oecm_total_gpd_area_total_km2 / chile_gpd_area_total_km2) * 100
print(f"Superficie Chile: {chile_gpd_area_total_km2:,.0f} km²")
print(f"Superficie Áreas Protegidas: {oecm_total_gpd_area_total_km2:,.0f} km²")
print(f"% protegido: {porcentaje:.2f}%")


Superficie Chile: 4,324,229 km²
Superficie Áreas Protegidas: 255,904 km²
% protegido: 5.92%


## Cobertura por Ecosistema (EFG) y Reino (pendiente capa EFG)

Pendiente

##  Cobertura de Sitios Clave (KBA Proxy):

Calcular el porcentaje medio de la superficie de Sitios Prioritarios que está cubierta por AP y/u OECM.

### KBA Proxy Áreas Protegidas
Intersección Areas Protegidas con Sitios Prioritarios

In [26]:
sitios_prioritarios_areas_protegidas_gpd = gpd.overlay(sitios_prioritarios_capa_gpd, areas_protegidas_capa_gpd, how="intersection", keep_geom_type=False)
sitios_prioritarios_areas_protegidas_gpd["geometry"] = sitios_prioritarios_areas_protegidas_gpd.geometry.make_valid()
sitios_prioritarios_areas_protegidas_gpd_union = sitios_prioritarios_areas_protegidas_gpd.geometry.union_all()
sitios_prioritarios_areas_protegidas_gpd_area_total_m2 = geodesic_area_m2(sitios_prioritarios_areas_protegidas_gpd_union)
sitios_prioritarios_areas_protegidas_gpd_area_total_ha = sitios_prioritarios_areas_protegidas_gpd_area_total_m2 / 10_000
sitios_prioritarios_areas_protegidas_gpd_area_total_km2 = sitios_prioritarios_areas_protegidas_gpd_area_total_ha / 100
porcentaje = (sitios_prioritarios_areas_protegidas_gpd_area_total_km2 / sitios_prioritarios_capa_gpd_area_total_km2) * 100
print(f"Superficie Áreas Protegidas en Sitios Prioritarios: {sitios_prioritarios_areas_protegidas_gpd_area_total_km2:,.0f} km²")
print(f"Superficie Sitios Prioritarios: {sitios_prioritarios_capa_gpd_area_total_km2:,.0f} km²")
print(f"% protegido: {porcentaje:.2f}%")

Superficie Áreas Protegidas en Sitios Prioritarios: 78,267 km²
Superficie Sitios Prioritarios: 174,645 km²
% protegido: 44.81%


### KBA Proxy OECM
Intersección OECM con Sitios Prioritarios

In [28]:
sitios_prioritarios_oecm_gpd = gpd.overlay(sitios_prioritarios_capa_gpd, oecm_total_gpd, how="intersection", keep_geom_type=False)
sitios_prioritarios_oecm_gpd["geometry"] = sitios_prioritarios_oecm_gpd.geometry.make_valid()
sitios_prioritarios_oecm_gpd_union = sitios_prioritarios_areas_protegidas_gpd.geometry.union_all()
sitios_prioritarios_oecm_gpd_area_total_m2 = geodesic_area_m2(sitios_prioritarios_oecm_gpd_union)
sitios_prioritarios_oecm_gpd_area_total_ha = sitios_prioritarios_oecm_gpd_area_total_m2 / 10_000
sitios_prioritarios_oecm_gpd_area_total_km2 = sitios_prioritarios_oecm_gpd_area_total_ha / 100
porcentaje = (sitios_prioritarios_oecm_gpd_area_total_km2 / sitios_prioritarios_capa_gpd_area_total_km2) * 100
print(f"Superficie Áreas Protegidas en Sitios Prioritarios: {sitios_prioritarios_oecm_gpd_area_total_km2:,.0f} km²")
print(f"Superficie Sitios Prioritarios: {sitios_prioritarios_capa_gpd_area_total_km2:,.0f} km²")
print(f"% protegido: {porcentaje:.2f}%")

Superficie Áreas Protegidas en Sitios Prioritarios: 78,267 km²
Superficie Sitios Prioritarios: 174,645 km²
% protegido: 44.81%


# JUPYTERBOOK

### Objetivo del Indicador
Medir el porcentaje de superficie cubierta por Áreas Protegidas (AP) y Otras Medidas de Conservación Eficaces Basadas en Áreas (OECM), desagregado según criterios.

Acceso a metadatos oficiales: https://www.gbf-indicators.org/metadata/headline/3-1

## Fuentes de Datos y Descripción

### Fuente principal (AP y OECM)
Capas espaciales de **Áreas Protegidas** e **iniciativas de conservación privadas**.

- **Visor (Áreas protegidas y OECM):** https://apps.mma.gob.cl/visorsimbio

#### Descargar capas (ArcGIS FeatureServer)
- **Áreas protegidas:** https://arcgis.mma.gob.cl/server/rest/services/SIMBIO/SIMBIO_AP/FeatureServer/0  
- **Otras designaciones:** https://arcgis.mma.gob.cl/server/rest/services/SIMBIO/SIMBIO_AP/FeatureServer/1  
- **Conservación privada:** https://arcgis.mma.gob.cl/server/rest/services/SIMBIO/SIMBIO_AP/FeatureServer/3  

---

### Humedales urbanos
Disponibles en el visor SIMBIO, categoría **Ecosistemas acuáticos**.

- **Visor:** https://apps.mma.gob.cl/visorsimbio  
- **Capa (ArcGIS FeatureServer):** https://arcgis.mma.gob.cl/server/rest/services/SIMBIO/SIMBIO_HUMEDALES/FeatureServer/1  

---

### Espacios Costeros Marinos para Pueblos Originarios (ECMPO)
- **Geoportal SUBPESCA:** https://geoportal.subpesca.cl/portal/home/item.html?id=20ca1b00d78a4c858724775d0a083a5e


In [33]:
import pandas as pd
from IPython.display import display, HTML

tabla_areas_fuentes = pd.DataFrame([
    {"Capa":"Áreas protegidas", "Valor": areas_protegidas_total_gpd_area_total_km2, "Unidad": "km²"},
    {"Capa":"Otras designaciones", "Valor": otras_designaciones_capa_gpd_area_total_km2, "Unidad": "km²"},
    {"Capa":"Sitios prioritarios", "Valor": sitios_prioritarios_capa_gpd_area_total_km2, "Unidad": "km²"},
    {"Capa":"Conservación privada", "Valor": conservacion_privada_capa_gpd_area_total_km2, "Unidad": "km²"},
    {"Capa":"Humedales urbanos", "Valor": humedales_gpd_area_total_km2, "Unidad": "km²"},  # si venía en m²
])

tabla_areas_fuentes["Valor"] = tabla_areas_fuentes["Valor"].map(lambda x: f"{x:,.0f}")

display(HTML(tabla_areas_fuentes.to_html(index=False)))


Capa,Valor,Unidad
Áreas protegidas,"1,995,850",km²
Otras designaciones,"174,645",km²
Sitios prioritarios,"174,645",km²
Conservación privada,"1,449,068,924",km²
Humedales urbanos,168,km²


## Procesamiento de Datos

El procesamiento se centra en la **integración**, **limpieza** y la **correcta identificación** de todas las figuras de conservación y los sitios de importancia.

### A. Filtro e identificación de figuras de conservación (columna `designación`)

**Áreas protegidas (AP):**
- Parque Nacional  
- Parque Marino  
- Reserva Nacional  
- Reserva Marina  
- Reserva Forestal  
- Monumento Natural  
- Santuarios de la Naturaleza  
- Área de Conservación de Múltiples Usos  
- Humedales urbanos  

**Otras medidas eficaces de conservación (OECM):**
- Paisaje de conservación  
- Conservación Privada y Comunitaria  
- Bien nacional protegido  
- Espacios Costeros Marinos para Pueblos Originarios (ECMPO)

**Exclusión:**
- Excluir la designación **“Reserva de la Biósfera”**.

---

### B. Limpieza de geometría y consistencia

- **Limpieza topológica:** eliminar geometrías y atributos duplicados o redundantes en la capa espacial de **AP/OECM**.

---

### C. Definición de KBAs (proxy)

Se utilizará la capa de **Sitios Prioritarios** como la mejor representación de las **Áreas de Particular Importancia para la Biodiversidad (KBA)** para el componente de **calidad de la cobertura**:

- https://lineasdebasepublicas.mma.gob.cl/datos_abiertos/group/areas-protegidas-sitios-prioritarios?organization=nacional&res_format=SHP

---

## 3. Desagregaciones Requeridas

El indicador final se debe presentar desglosado para cubrir los requisitos de la **Meta 3**.

### a) Cobertura por tipo (AP vs. OECM)

- Calcular el **porcentaje de superficie total** cubierto por **Áreas Protegidas (AP)**.  
- Calcular el **porcentaje de superficie total** cubierto por **OECM**.

---

### b) Cobertura por ecosistema (EFG) y reino (pendiente capa EFG)

- **Análisis de intersección:** intersectar la capa final **AP/OECM** con la **Capa EFG**.
- Reportar el **porcentaje de cobertura de AP/OECM** para cada **Reino Geográfico**:
  - Terrestre
  - Aguas Continentales
  - Marino/Costero
- Reportar el **porcentaje de cobertura** para cada **EFG (Bioma/Ecosistema)** individualmente.

---

### c) Cobertura de sitios clave (KBA proxy)

- Calcular el **porcentaje medio** de la superficie de **Sitios Prioritarios** que está cubierta por **AP y/u OECM**.

---

### d) Eficacia de la gestión

- No se ha realizado una evaluación formal de eficacia en la gestión a nivel nacional.

**Nota:**  
Se debe definir un **proxy** basado en atributos de gestión disponibles (ej. existencia o vigencia de planes de manejo) **o** reportar el componente como:
- **“No Disponible”**, o
- **“Detalles Aún en Desarrollo”**  
(según el metadato CBD).

---

### e) Cobertura por gobernanza

- Realizar la desagregación espacial del total de **AP/OECM** según tipo de administración, utilizando atributos disponibles:
  - Gobierno
  - Organizaciones Privadas
  - Pueblos Indígenas / Comunidades Locales
  - Compartidas

**Supuestos / validaciones:**
- Para **OECM** se asume que **todas son privadas**.
- Para **AP**, la base de datos incluye una hoja llamada **“propiedad”**, donde se indica si son **privadas o públicas**.  
  Esta información debe ser **posteriormente validada**.

---

## Matriz de clasificación (Designación / Tipo / Gobernanza)

| Designación | Tipo | Clasificación de gobernanza |
|---|---|---|
| Parque Nacional | Área Protegida | Gobierno |
| Reserva Nacional | Área Protegida | Gobierno |
| Reserva Forestal | Área Protegida | Gobierno |
| Monumento Natural | Área Protegida | Gobierno |
| Parque Marino | Área Protegida | Gobierno |
| Reserva Marina | Área Protegida | Gobierno |
| Santuario de la Naturaleza | Área Protegida | Privada (mayoría) / Gobierno (algunos) |
| Humedal Urbano | Área Protegida | Gobierno (Municipal/MMA) |
| ECMPO | OECM (Potencial) | Pueblos Indígenas |
